# Day 18：OOF Probability Ensemble and Stable Threshold Selection

本 notebook 复盘 Day18 的 OOF 概率集成实验。Day18 只使用 official train 内部的 OOF 预测，不使用 official test；本轮不做权重搜索，也不使用 Day15 tuned models。


## 1. 为什么做 probability ensemble

Day14 显示 `median_all_structural_all` 是当前较稳的主模型；`median_with_selected_missing_indicators` 和 `prefix_zero_rate` 更偏 recall 补漏，但 FP 更高。Day18 的问题是：这些模型是否能在 OOF 上互相补回漏报，并且不明显增加总成本。

`median_with_selected_missing_indicators` 对应历史 `median_with_indicator` 方向的更严格 OOF 版本：原始数值特征中位数填充 + fold_train 内筛选 selected missing indicators，fold_valid 只做 transform。

注意：历史 official test 数字只用于背景说明，本轮不根据它们调整权重或阈值。


In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from scania_aps.config import get_config

cfg = get_config(PROJECT_ROOT / 'config' / 'config.yaml')
cfg.oof_ensemble['stage']


'day18_oof_probability_ensemble'

## 2. 读取 Day18 输出

如果尚未生成结果，先在项目根目录运行：`python scripts/16_oof_probability_ensemble.py`。


In [2]:
base_best = pd.read_csv(cfg.metrics_dir / 'day18_oof_base_best_summary.csv')
ensemble_best = pd.read_csv(cfg.metrics_dir / 'day18_oof_ensemble_best_summary.csv')
fn_overlap = pd.read_csv(cfg.tables_dir / 'day18_oof_fn_overlap_summary.csv')
fp_overlap = pd.read_csv(cfg.tables_dir / 'day18_oof_fp_overlap_summary.csv')
recommendations = pd.read_csv(cfg.tables_dir / 'day18_official_test_candidate_recommendations.csv')
base_best.shape, ensemble_best.shape, recommendations.shape


((3, 19), (84, 20), (84, 22))

## 3. Base OOF strategy 结果

三类 base strategy 使用同一组 OOF folds，便于逐样本 overlap analysis。


In [3]:
base_best[['strategy', 'threshold', 'precision', 'recall', 'f2', 'average_precision', 'fp', 'fn', 'total_cost']]


,strategy,threshold,precision,recall,f2,average_precision,fp,fn,total_cost
0,median_all_structural_all,0.09,0.301310,0.966,0.670275,0.871098,2240,34,39400
1,median_with_selected_missing_indicators,0.07,0.275774,0.971,0.645526,0.869068,2550,29,40000
2,median_all_prefix_zero_rate,0.14,0.348701,0.953,0.707708,0.871301,1780,47,41300


## 4. FN / FP overlap analysis

这里重点看 `median_all_structural_all` 漏掉的正类，indicator / prefix zero 是否能补回，以及补回时带来多少额外 FP。


In [4]:
fn_overlap


,reference_strategy,reference_fn_count,indicator_rescued_fn,prefixzero_rescued_fn,either_rescued_fn,all_models_fn,structural_only_fn,indicator_extra_fp_vs_reference,prefixzero_extra_fp_vs_reference
0,median_all_structural_all,34,7,1,7,27,7,416,49


In [5]:
fp_overlap


,fp_overlap_type,sample_count
0,all_three_fp,1720
1,structural_only_fp,95
2,indicator_only_fp,371
3,prefixzero_only_fp,4
4,structural_indicator_fp,414
5,structural_prefixzero_fp,11
6,indicator_prefixzero_fp,45


## 5. 12 个 ensemble recipe 的 cost_min 对比

这些 recipe 都是预先固定的概率平均或 rank 平均，没有做权重搜索。`diagnostic` 方案只用于观察上限，不进入最终候选。


In [6]:
cost_min = ensemble_best[ensemble_best['threshold_selection_rule'].eq('cost_min')]
cost_min[['ensemble_name', 'ensemble_group', 'threshold', 'precision', 'recall', 'f2', 'average_precision', 'fp', 'fn', 'total_cost']].sort_values('total_cost')


,ensemble_name,ensemble_group,threshold,precision,recall,f2,average_precision,fp,fn,total_cost
0,mean_structural_indicator,auxiliary,0.14,0.342500,0.959,0.705147,0.871843,1841,41,38910
1,weighted_70_20_10,main,0.10,0.310589,0.965,0.678908,0.872562,2142,35,38920
2,weighted_60_25_15,main,0.10,0.310190,0.965,0.678526,0.872657,2146,35,38960
3,mean_three_models,auxiliary,0.13,0.334728,0.960,0.698893,0.872759,1908,40,39080
4,weighted_80_20_prefixzero,auxiliary,0.10,0.310667,0.964,0.678587,0.872392,2139,36,39390
5,mean_structural_prefixzero,diagnostic,0.11,0.320560,0.962,0.687045,0.873175,2039,38,39390
6,structural_all_single,main,0.09,0.301310,0.966,0.670275,0.871098,2240,34,39400
7,max_three_models,diagnostic,0.16,0.330692,0.960,0.695350,0.871832,1943,40,39430
8,weighted_70_30_indicator,main,0.15,0.352248,0.956,0.711945,0.872055,1758,44,39580
9,weighted_80_20_indicator,main,0.15,0.352248,0.956,0.711945,0.871835,1758,44,39580


## 6. Recall floor / FN floor 规则

Day18 使用较温和的 recall/FN floor：`recall_floor_960/965/970` 和 `fn_floor_35/40/45`。这些是业务约束观察，不是为了刷模型分数。


In [7]:
rules = ['recall_floor_960', 'recall_floor_965', 'recall_floor_970', 'fn_floor_35', 'fn_floor_40', 'fn_floor_45']
ensemble_best[ensemble_best['threshold_selection_rule'].isin(rules)][['threshold_selection_rule', 'ensemble_name', 'ensemble_group', 'threshold', 'recall', 'fp', 'fn', 'total_cost', 'constraint_satisfied']].sort_values(['threshold_selection_rule', 'total_cost']).groupby('threshold_selection_rule').head(3)


,threshold_selection_rule,ensemble_name,ensemble_group,threshold,recall,fp,fn,total_cost,constraint_satisfied
12,fn_floor_35,weighted_70_20_10,main,0.10,0.965,2142,35,38920,True
13,fn_floor_35,weighted_60_25_15,main,0.10,0.965,2146,35,38960,True
14,fn_floor_35,structural_all_single,main,0.09,0.966,2240,34,39400,True
24,fn_floor_40,weighted_70_20_10,main,0.10,0.965,2142,35,38920,True
25,fn_floor_40,weighted_60_25_15,main,0.10,0.965,2146,35,38960,True
26,fn_floor_40,mean_three_models,auxiliary,0.13,0.960,1908,40,39080,True
36,fn_floor_45,mean_structural_indicator,auxiliary,0.14,0.959,1841,41,38910,True
37,fn_floor_45,weighted_70_20_10,main,0.10,0.965,2142,35,38920,True
38,fn_floor_45,weighted_60_25_15,main,0.10,0.965,2146,35,38960,True
48,recall_floor_960,weighted_70_20_10,main,0.10,0.965,2142,35,38920,True


## 7. Day19 official test candidate 筛选

进入 Day19 的候选必须相对 `structural_all_single` 同时满足：OOF cost 至少下降 3%、FN 至少减少 3 个、FP 增加可控，且不能是 diagnostic / max-three 方案。


In [8]:
recommendations[['ensemble_name', 'threshold_selection_rule', 'threshold', 'oof_recall', 'oof_fp', 'oof_fn', 'oof_total_cost', 'delta_fp', 'delta_fn', 'cost_reduction_pct', 'recommend_for_day19', 'recommendation_reason']].head(20)


,ensemble_name,threshold_selection_rule,threshold,oof_recall,oof_fp,oof_fn,oof_total_cost,delta_fp,delta_fn,cost_reduction_pct,recommend_for_day19,recommendation_reason
0,mean_structural_indicator,cost_min,0.14,0.959,1841,41,38910,-399,7,0.012437,False,FN increased by 7; cost reduction 0.012 < 0.030
1,mean_structural_indicator,fn_floor_45,0.14,0.959,1841,41,38910,-399,7,0.012437,False,FN increased by 7; cost reduction 0.012 < 0.030
2,mean_three_models,cost_min,0.13,0.960,1908,40,39080,-332,6,0.008122,False,FN increased by 6; cost reduction 0.008 < 0.030
3,mean_three_models,fn_floor_40,0.13,0.960,1908,40,39080,-332,6,0.008122,False,FN increased by 6; cost reduction 0.008 < 0.030
4,mean_three_models,fn_floor_45,0.13,0.960,1908,40,39080,-332,6,0.008122,False,FN increased by 6; cost reduction 0.008 < 0.030
5,mean_three_models,recall_floor_960,0.13,0.960,1908,40,39080,-332,6,0.008122,False,FN increased by 6; cost reduction 0.008 < 0.030
6,weighted_80_20_prefixzero,cost_min,0.10,0.964,2139,36,39390,-101,2,0.000254,False,FN increased by 2; cost reduction 0.000 < 0.030
7,weighted_80_20_prefixzero,fn_floor_40,0.10,0.964,2139,36,39390,-101,2,0.000254,False,FN increased by 2; cost reduction 0.000 < 0.030
8,weighted_80_20_prefixzero,fn_floor_45,0.10,0.964,2139,36,39390,-101,2,0.000254,False,FN increased by 2; cost reduction 0.000 < 0.030
9,weighted_80_20_prefixzero,recall_floor_960,0.10,0.964,2139,36,39390,-101,2,0.000254,False,FN increased by 2; cost reduction 0.000 < 0.030


## 8. 中文结论

- `median_with_selected_missing_indicators` 能补回一部分 structural_all 的 OOF FN，但同时带来大量额外 FP；prefix zero 的补漏更弱。
- 多数 ensemble 的 OOF 成本改善很小，主要来自降低 FP，而不是稳定减少 FN。
- `weighted_rank_60_25_15` 能把 FN 降到 30，但 FP 增加到 2577，total cost 反而更高。
- 根据当前筛选规则，没有方案满足进入 Day19 official test 的条件。因此 Day18 不推荐强行做 official test。
- 如果继续增强，更合理的方向是解释性分析、SQL 业务深化、README 收尾，或单独深化更精细的 histogram/bin projection，而不是继续做概率平均或权重搜索。
